# Programmatic Evaluator Management via REST API

## Goal

This notebook demonstrates how to **programmatically create, manage, and bind evaluators** to your GenAI applications using Fiddler's public REST API. This enables automated deployment pipelines where a base set of evaluators is provisioned at application creation time, and developers can add additional evaluators from an approved list.

## Key Concepts

Evaluator management follows three steps:

1. **Select an evaluator type** -- Fiddler offers built-in trust models (safety, PII), RAG metrics (answer relevance), and custom LLM-as-a-Judge evaluators. Use the config options API to discover what's available.
2. **Create an evaluator (org-scoped)** -- Define the evaluator at the organization level. It describes *what* to evaluate, not *where*. For custom judges, this includes a `prompt_spec` with your evaluation prompt and expected output fields. For built-in types, minimal or no configuration is needed.
3. **Bind it with a rule (app-scoped)** -- An evaluator rule connects the evaluator to a specific application, maps span attributes to the evaluator's inputs, and optionally filters which spans get evaluated.

## What You'll Learn

1. Discover available evaluator types and their configuration schemas
2. Look up LLM Gateway providers (for evaluators that require external LLMs)
3. Create evaluators: custom LLM-as-a-Judge, RAG metrics, and Fiddler Trust Models
4. Bind evaluators to applications via evaluator rules
5. List, update, and delete evaluators and rules

### Porting Existing Evaluators

If you already have evaluation prompts from pre-production (e.g., calling an LLM to score responses for quality, compliance, or relevance), you can port them directly into Fiddler:

1. Take your existing evaluation prompt
2. Replace the dynamic parts (user query, LLM response, context, etc.) with `{{placeholder}}` syntax
3. Define the expected output fields (scores, labels, reasoning) with their types
4. Wrap it all in a `prompt_spec` and create an evaluator via the API
5. Bind it to your application with an evaluator rule that maps span attributes to the placeholders

Step 3a walks through this in detail with multiple examples.

## Prerequisites

- A Fiddler environment with API access
- An API token (from **Settings > Credentials**)
- At least one GenAI application already created in Fiddler
- For LLM-based evaluators: an LLM credential configured in **Settings > LLM Gateway**

## Rate Limits

These endpoints are rate-limited to **10 requests/second** and **300 requests/minute** per client. If you exceed these limits, you'll receive a `429` response with `Retry-After` and `X-RateLimit-*` headers indicating when you can retry.

---
## 0. Setup

In [ ]:
%pip install -q fiddler-client

import json
import time

import fiddler as fdl
import requests

In [ ]:
# -- Connection settings --------------------------------------------------
URL = ''  # e.g. 'https://your-org.fiddler.ai'
TOKEN = ''  # From Settings > Credentials

assert URL != '', 'Please set your Fiddler URL'
assert TOKEN != '', 'Please set your Fiddler API token'

# -- Connect via SDK (for project/application context) --------------------
fdl.init(url=URL, token=TOKEN)

# -- REST API setup -------------------------------------------------------
BASE_URL = URL.rstrip('/')
HEADERS = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {TOKEN}',
}

In [ ]:
def fiddler_api(
    method: str,
    endpoint: str,
    payload: dict = None,
    params: dict = None,
    max_retries: int = 3,
) -> dict:
    """Call a Fiddler REST API endpoint with automatic retry on rate limits.

    Args:
        method: HTTP method ('GET', 'POST', 'PATCH', 'DELETE').
        endpoint: API path (e.g. '/v3/evaluators').
        payload: JSON body for POST/PATCH requests.
        params: Query parameters.
        max_retries: Number of retries on 429 responses.

    Returns:
        Parsed JSON response.

    Raises:
        Exception: On non-2xx responses (after retries for 429s).
    """
    url = f'{BASE_URL}{endpoint}'
    for attempt in range(max_retries + 1):
        response = requests.request(
            method=method,
            url=url,
            headers=HEADERS,
            json=payload,
            params=params,
        )
        if response.status_code == 429:
            retry_after = int(response.headers.get('Retry-After', 1))
            remaining = response.headers.get('X-RateLimit-Remaining', '?')
            print(
                f'  Rate limited (attempt {attempt + 1}/{max_retries + 1}). '
                f'Remaining: {remaining}. Retrying in {retry_after}s...'
            )
            time.sleep(retry_after)
            continue
        break

    data = response.json()
    if response.status_code >= 400:
        raise Exception(
            f'{method} {endpoint} failed ({response.status_code}): '
            f'{json.dumps(data, indent=2)}'
        )
    return data

### Application and Provider IDs

Set the IDs for your target application and LLM provider credentials. You can find these in the Fiddler UI or by querying the APIs below.

In [ ]:
# -- Replace these with your actual IDs ------------------------------------
APPLICATION_ID = '<your-application-uuid>'  # GenAI application to bind rules to
LLM_MODEL_ID = '<your-llm-model-uuid>'  # From GET /v3/llm-gateway/providers
LLM_CREDENTIAL_ID = '<your-llm-credential-uuid>'  # From GET /v3/llm-gateway/providers

---
## 1. Discover Available Evaluator Types

Before creating evaluators, use `GET /v3/evaluators/get-config-options` to see which evaluator types are available and what configuration each one expects.

In [ ]:
config_response = fiddler_api('GET', '/v3/evaluators/get-config-options')
config_schemas = config_response['data']['config_schemas']

print(f'Available evaluator types ({len(config_schemas)}):\n')
for schema in config_schemas:
    print(f"  - {schema['enrichment_name']:30s} ({schema['enrichment_display_name']})")

In [ ]:
# Inspect the config schema for a specific evaluator type
evaluator_type = 'llm_as_a_judge'  # Change this to inspect other types

for schema in config_schemas:
    if schema['enrichment_name'] == evaluator_type:
        print(f"Config schema for '{evaluator_type}':")
        print(json.dumps(schema['enrichment_config'], indent=2))
        break

---
## 2. Look Up LLM Gateway Providers

Evaluators that use external LLMs (like `answer_relevance_v2`, `rag_faithfulness`, or `llm_as_a_judge`) require a `model_id` and `credential_id` from your LLM Gateway configuration.

Use `GET /v3/llm-gateway/providers` to list configured providers and extract the UUIDs you need.

In [ ]:
providers_response = fiddler_api('GET', '/v3/llm-gateway/providers')
providers = providers_response['data']['items']

for provider in providers:
    print(f"Provider: {provider['provider']}")
    for cred in provider.get('credentials', []):
        print(f"  Credential: {cred['name']} (ID: {cred['uuid']})")
    for model in provider.get('models', []):
        print(f"  Model:      {model['name']} (ID: {model['uuid']})")
    print()

---
## 3. Create Evaluators

Evaluators are **org-scoped** -- they define *what* to evaluate, not *where*. You bind them to specific applications in Step 4 using evaluator rules.

### Key fields

| Field | Required | Description |
|-------|----------|-------------|
| `name` | Yes | Any descriptive name you choose (1-256 characters, e.g. `'Prompt Safety'`) |
| `enrichment_name` | Yes | Must be an exact `enrichment_name` value from Step 1 (e.g. `'ftl_prompt_safety'`, `'answer_relevance_v2'`) |
| `enrichment_config` | Depends | Configuration object whose shape is defined by the `enrichment_config` schema from Step 1 |

**How to find the right values:** Run Step 1 first. The `enrichment_name` column gives you the exact string to use, and the config schema tells you what `enrichment_config` fields are required (if any).

Below are three patterns, starting with Custom LLM-as-a-Judge since that is the most flexible and commonly used for porting existing evaluation logic into Fiddler.

### 3a. Custom LLM-as-a-Judge (requires LLM Gateway)

This is the most flexible evaluator type -- use it to port existing evaluation prompts from your pre-production workflow into Fiddler.

Every custom judge evaluator is defined by a **`prompt_spec`** with two parts:

- **`prompt_template`** -- The message(s) sent to the judge LLM. Uses standard chat format (`role` + `content`). Dynamic values from your spans are injected via `{{placeholder}}` syntax. The placeholder names you choose here become the evaluator's input fields, which you later map to span attributes in the evaluator rule (Step 4).

- **`output_fields`** -- The structured response you expect back from the judge LLM. Each field becomes a metric column in Fiddler that you can chart, alert on, and use in RCA. Supported types: `number`, `string`, `boolean`.

Below are three examples showing different patterns. **Run Example 1 to get started.** Examples 2 and 3 show additional patterns you can reference later.

#### Example 1: Simple quality scoring (two inputs, numeric + string output)

A single `user` message with two placeholders. The judge returns a numeric score and a text explanation.

In [ ]:
# 'name' is freeform -- call it whatever you want.
# 'enrichment_name' must match a value from Step 1 (GET /v3/evaluators/get-config-options).
# llm_as_a_judge requires LLM Gateway credentials (model_id + credential_id from Step 2).
custom_judge_evaluator = fiddler_api('POST', '/v3/evaluators', payload={
    'name': 'Response Quality Judge',
    'enrichment_name': 'llm_as_a_judge',
    'enrichment_config': {
        'model_id': LLM_MODEL_ID,
        'credential_id': LLM_CREDENTIAL_ID,
        'prompt_spec': {
            'prompt_template': [
                {
                    'role': 'user',
                    'content': (
                        'Evaluate the quality of the following AI response '
                        'to the user query.\n\n'
                        'User Query: {{input}}\n\n'
                        'AI Response: {{output}}\n\n'
                        'Rate the response on completeness and accuracy.'
                    ),
                }
            ],
            'output_fields': {
                'quality_score': {
                    'type': 'number',
                    'description': 'Overall quality score from 0 to 1',
                },
                'reasoning': {
                    'type': 'string',
                    'description': 'Brief explanation of the score',
                },
            },
        },
    },
})

custom_judge_id = custom_judge_evaluator['data']['id']
print(f"Created custom judge evaluator: {custom_judge_id}")
print(json.dumps(custom_judge_evaluator['data'], indent=2))

#### Example 2: RAG groundedness check with system persona (three inputs, boolean output)

Uses a `system` message to set the judge's persona, and a `user` message with three placeholders (`{{question}}`, `{{context}}`, `{{answer}}`). The judge returns a boolean pass/fail and a string explanation.

This pattern is useful when you want a custom groundedness check with your own criteria, beyond what the built-in `rag_faithfulness` evaluator provides.

In [ ]:
groundedness_judge = fiddler_api('POST', '/v3/evaluators', payload={
    'name': 'Groundedness Judge',
    'enrichment_name': 'llm_as_a_judge',
    'enrichment_config': {
        'model_id': LLM_MODEL_ID,
        'credential_id': LLM_CREDENTIAL_ID,
        'prompt_spec': {
            'prompt_template': [
                {
                    'role': 'system',
                    'content': (
                        'You are a strict fact-checking evaluator. '
                        'Only mark a response as grounded if every claim '
                        'is directly supported by the provided context.'
                    ),
                },
                {
                    'role': 'user',
                    'content': (
                        'Question: {{question}}\n\n'
                        'Context provided to the AI:\n{{context}}\n\n'
                        'AI Response:\n{{answer}}\n\n'
                        'Is the response fully grounded in the context?'
                    ),
                }
            ],
            'output_fields': {
                'is_grounded': {
                    'type': 'boolean',
                    'description': 'True if every claim in the response is supported by the context',
                },
                'unsupported_claims': {
                    'type': 'string',
                    'description': 'List any claims not supported by the context, or empty if fully grounded',
                },
            },
        },
    },
})

groundedness_judge_id = groundedness_judge['data']['id']
print(f"Created groundedness judge: {groundedness_judge_id}")
print(json.dumps(groundedness_judge['data'], indent=2))

#### Example 3: Compliance check (single input, boolean + string output)

Sometimes you only need to evaluate the output itself, without the input query. This example uses a single placeholder (`{{output}}`) and returns a pass/fail compliance flag.

In [ ]:
compliance_judge = fiddler_api('POST', '/v3/evaluators', payload={
    'name': 'Compliance Judge',
    'enrichment_name': 'llm_as_a_judge',
    'enrichment_config': {
        'model_id': LLM_MODEL_ID,
        'credential_id': LLM_CREDENTIAL_ID,
        'prompt_spec': {
            'prompt_template': [
                {
                    'role': 'user',
                    'content': (
                        'Review the following AI-generated response for '
                        'compliance with financial services regulations.\n\n'
                        'Response: {{output}}\n\n'
                        'Check for: unauthorized financial advice, '
                        'unsubstantiated claims, missing disclaimers.'
                    ),
                }
            ],
            'output_fields': {
                'is_compliant': {
                    'type': 'boolean',
                    'description': 'True if the response meets compliance standards',
                },
                'violation_type': {
                    'type': 'string',
                    'description': 'Type of violation found, or empty if compliant',
                },
            },
        },
    },
})

compliance_judge_id = compliance_judge['data']['id']
print(f"Created compliance judge: {compliance_judge_id}")
print(json.dumps(compliance_judge['data'], indent=2))

### 3b. RAG Metric (requires LLM Gateway)

RAG evaluators like `answer_relevance_v2` and `rag_faithfulness` use an external LLM to judge response quality.

In [ ]:
answer_relevance_evaluator = fiddler_api('POST', '/v3/evaluators', payload={
    'name': 'Answer Relevance',
    'enrichment_name': 'answer_relevance_v2',
    'enrichment_config': {
        'model_id': LLM_MODEL_ID,
        'credential_id': LLM_CREDENTIAL_ID,
    },
})

answer_relevance_id = answer_relevance_evaluator['data']['id']
print(f"Created answer relevance evaluator: {answer_relevance_id}")
print(json.dumps(answer_relevance_evaluator['data'], indent=2))

### 3c. Fiddler Trust Model (no external LLM needed)

Fiddler's built-in trust models (e.g. `ftl_prompt_safety`, `pii`) run on Fiddler's infrastructure and don't require LLM Gateway credentials.

In [ ]:
# Trust models like ftl_prompt_safety need no enrichment_config.
safety_evaluator = fiddler_api('POST', '/v3/evaluators', payload={
    'name': 'Prompt Safety',
    'enrichment_name': 'ftl_prompt_safety',
})

safety_evaluator_id = safety_evaluator['data']['id']
print(f"Created safety evaluator: {safety_evaluator_id}")
print(json.dumps(safety_evaluator['data'], indent=2))

### 3d. List All Evaluators

View all evaluators in your organization. Useful for verifying what's been created or finding evaluator IDs.

In [ ]:
evaluators_response = fiddler_api('GET', '/v3/evaluators')
evaluators = evaluators_response['data']['items']

print(f'Evaluators in org ({len(evaluators)}):\n')
for ev in evaluators:
    print(f"  {ev['name']:40s} | {ev['enrichment_name']:25s} | ID: {ev['id']}")

### 3e. Get Evaluator Details

In [ ]:
evaluator_detail = fiddler_api('GET', f'/v3/evaluators/{safety_evaluator_id}')
print(json.dumps(evaluator_detail['data'], indent=2))

### 3f. Update an Evaluator

Use `PATCH` to update an evaluator's name or configuration.

In [ ]:
updated_evaluator = fiddler_api('PATCH', f'/v3/evaluators/{safety_evaluator_id}', payload={
    'name': 'Prompt Safety (Updated)',
})

print(f"Updated evaluator name: {updated_evaluator['data']['name']}")

---
## 4. Create Evaluator Rules

An evaluator rule **binds an evaluator to an application**. It specifies:
- Which evaluator to run
- Which span attributes map to the evaluator's input fields
- (Optional) Filters to only evaluate certain spans
- (Optional) Whether to backfill historical data

### How `mapped_input_keys` connects to `prompt_template`

The `{{placeholder}}` names you used in your `prompt_template` (Step 3a) become the **keys** in `mapped_input_keys`. The **values** are the span attribute paths where the actual data lives. For example, if your template has `{{question}}` and `{{answer}}`:

```python
'mapped_input_keys': {
    'question': 'fiddler.contents.gen_ai.llm.input.user',   # {{question}} gets the user prompt
    'answer': 'fiddler.contents.gen_ai.llm.output',         # {{answer}} gets the LLM response
}
```

### Discovering Available Attributes

Use `POST /v3/evaluator-rules/map-input-keys` to see what span attributes exist in your application and what input fields the evaluator expects. If an expected attribute is missing from this list, it means your application's spans are not currently capturing that data -- you'll need to update your instrumentation to include it, or map to a different attribute that is available.

In [ ]:
# Pass the evaluator you want to bind -- the response will show
# the evaluator's expected input fields and your app's available span attributes.
input_keys = fiddler_api('POST', '/v3/evaluator-rules/map-input-keys', payload={
    'application_id': APPLICATION_ID,
    'evaluator_id': custom_judge_id,
})

print('Available attribute names in your application:')
print(json.dumps(input_keys['data'], indent=2))

### 4a. Create a Rule for a Custom Judge (Two Inputs)

Bind the quality judge from Example 1 (Step 3a) to all spans. The `mapped_input_keys` connect the template's `{{input}}` and `{{output}}` placeholders to actual span attributes.

In [ ]:
# The keys here ('input', 'output') must match the {{placeholder}} names
# in the evaluator's prompt_template from Step 3a Example 1.
judge_rule_simple = fiddler_api('POST', '/v3/evaluator-rules', payload={
    'name': 'Quality Judge - All Spans',
    'application_id': APPLICATION_ID,
    'evaluator_id': custom_judge_id,
    'mapped_input_keys': {
        'input': 'fiddler.contents.gen_ai.llm.input.user',
        'output': 'fiddler.contents.gen_ai.llm.output',
    },
    'backfill': False,
})

judge_rule_simple_id = judge_rule_simple['data']['id']
print(f"Created rule: {judge_rule_simple_id}")
print(json.dumps(judge_rule_simple['data'], indent=2))

### 4b. Create a Rule with Three Inputs and Filters

Bind the groundedness judge from Example 2 (Step 3a) to LLM spans only. This demonstrates mapping three placeholders (`{{question}}`, `{{context}}`, `{{answer}}`) and filtering by span type.

Filters support `SpanName` and `SpanType` fields with operators like `equal`, `in`, `not_equal`, etc.

In [ ]:
# Three-input mapping: each key matches a {{placeholder}} in the
# groundedness judge's prompt_template from Step 3a Example 2.
groundedness_rule = fiddler_api('POST', '/v3/evaluator-rules', payload={
    'name': 'Groundedness - LLM Spans Only',
    'application_id': APPLICATION_ID,
    'evaluator_id': groundedness_judge_id,
    'mapped_input_keys': {
        'question': 'fiddler.contents.gen_ai.llm.input.user',
        'context': 'fiddler.contents.gen_ai.llm.input.system',
        'answer': 'fiddler.contents.gen_ai.llm.output',
    },
    'filters': {
        'condition': 'AND',
        'rules': [
            {
                'field': 'SpanType',
                'operator': 'in',
                'value': ['llm'],
            },
        ],
    },
    'backfill': False,
})

groundedness_rule_id = groundedness_rule['data']['id']
print(f"Created filtered rule: {groundedness_rule_id}")
print(json.dumps(groundedness_rule['data'], indent=2))

### 4c. Create a Rule with Backfill

Set `backfill: true` to evaluate historical spans. Optionally provide `backfill_start_time` (ISO 8601) to limit the backfill window.

In [ ]:
relevance_rule = fiddler_api('POST', '/v3/evaluator-rules', payload={
    'name': 'Answer Relevance - With Backfill',
    'application_id': APPLICATION_ID,
    'evaluator_id': answer_relevance_id,
    'mapped_input_keys': {
        'user_query': 'fiddler.contents.gen_ai.llm.input.user',
        'rag_response': 'fiddler.contents.gen_ai.llm.output',
    },
    'backfill': True,
    'backfill_start_time': '2025-01-01T00:00:00Z',
})

relevance_rule_id = relevance_rule['data']['id']
print(f"Created rule with backfill: {relevance_rule_id}")

### 4d. List Evaluator Rules

List all rules, optionally filtered by application.

In [ ]:
# List all rules for a specific application
rules_filter = json.dumps({
    'condition': 'AND',
    'rules': [
        {
            'field': 'application_id',
            'operator': 'equal',
            'value': APPLICATION_ID,
        }
    ],
})

rules_response = fiddler_api('GET', '/v3/evaluator-rules', params={'filter': rules_filter})
rules = rules_response['data']['items']

print(f'Evaluator rules for application ({len(rules)}):\n')
for rule in rules:
    print(
        f"  {rule['name']:40s} | "
        f"evaluator: {rule.get('evaluator_name', 'N/A'):25s} | "
        f"enabled: {rule.get('enabled', 'N/A')}"
    )

### 4e. Update an Evaluator Rule

In [ ]:
updated_rule = fiddler_api('PATCH', f'/v3/evaluator-rules/{judge_rule_simple_id}', payload={
    'name': 'Quality Judge - All Spans (Updated)',
})

print(f"Updated rule name: {updated_rule['data']['name']}")

---
## 5. Check Backfill Status

If you created rules with `backfill: true`, you can check the status of backfill jobs.

In [ ]:
backfills_response = fiddler_api('GET', '/v3/genai-enrichment-backfills')
backfills = backfills_response['data']['items']

print(f'Backfill jobs ({len(backfills)}):\n')
for bf in backfills:
    print(json.dumps(bf, indent=2))

---
## 6. Automation Pattern: Deploy Custom Judges to an Application

This example shows how to programmatically deploy a set of custom LLM-as-a-Judge evaluators to an application -- the core use case for teams porting existing evaluation logic from pre-production into Fiddler.

Define your evaluator set once, then apply it to any application with a single function call.

In [ ]:
# Define a registry of custom judge evaluators to deploy.
# Each entry specifies the evaluator ID, a rule name, and input mappings.
STANDARD_JUDGE_RULES = [
    {
        'name': 'Quality Judge - LLM Spans',
        'evaluator_id': '<your-quality-judge-evaluator-id>',
        'mapped_input_keys': {
            'input': 'fiddler.contents.gen_ai.llm.input.user',
            'output': 'fiddler.contents.gen_ai.llm.output',
        },
        'filters': {
            'condition': 'AND',
            'rules': [
                {'field': 'SpanType', 'operator': 'in', 'value': ['llm']},
            ],
        },
    },
    {
        'name': 'Compliance Judge - All Spans',
        'evaluator_id': '<your-compliance-judge-evaluator-id>',
        'mapped_input_keys': {
            'input': 'fiddler.contents.gen_ai.llm.input.user',
            'output': 'fiddler.contents.gen_ai.llm.output',
        },
    },
]


def deploy_judges(application_id: str, rules: list) -> list:
    """Deploy a set of custom judge evaluator rules to an application.

    Args:
        application_id: Target GenAI application UUID.
        rules: List of rule config dicts (each must have evaluator_id,
               name, and mapped_input_keys).

    Returns:
        List of created rule responses.
    """
    created_rules = []
    for rule_config in rules:
        payload = {**rule_config, 'application_id': application_id, 'backfill': False}
        try:
            result = fiddler_api('POST', '/v3/evaluator-rules', payload=payload)
            print(f"  Created: {rule_config['name']} -> {result['data']['id']}")
            created_rules.append(result)
        except Exception as e:
            print(f"  Failed: {rule_config['name']} -> {e}")

    return created_rules


# -- Usage ----------------------------------------------------------------
# Uncomment to deploy:

# created = deploy_judges(
#     application_id=APPLICATION_ID,
#     rules=STANDARD_JUDGE_RULES,
# )
# print(f'\nDeployed {len(created)} evaluator rules.')

---
## 7. Cleanup

Delete evaluator rules and archive evaluators when no longer needed.

**Note:** You must delete all rules referencing an evaluator before you can archive it.

In [ ]:
# Delete evaluator rules
for rule_id, rule_name in [
    (judge_rule_simple_id, 'Quality judge rule'),
    (groundedness_rule_id, 'Groundedness rule'),
    (relevance_rule_id, 'Relevance rule'),
]:
    try:
        fiddler_api('DELETE', f'/v3/evaluator-rules/{rule_id}')
        print(f'  Deleted {rule_name}: {rule_id}')
    except Exception as e:
        print(f'  Failed to delete {rule_name}: {e}')

In [ ]:
# Archive evaluators
for eval_id, eval_name in [
    (custom_judge_id, 'Quality Judge evaluator'),
    (groundedness_judge_id, 'Groundedness Judge evaluator'),
    (compliance_judge_id, 'Compliance Judge evaluator'),
    (answer_relevance_id, 'Answer Relevance evaluator'),
    (safety_evaluator_id, 'Safety evaluator'),
]:
    try:
        fiddler_api('POST', f'/v3/evaluators/{eval_id}/archive')
        print(f'  Archived {eval_name}: {eval_id}')
    except Exception as e:
        print(f'  Failed to archive {eval_name}: {e}')

---
## API Reference

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/v3/evaluators/get-config-options` | GET | List available evaluator types and their config schemas |
| `/v3/llm-gateway/providers` | GET | List LLM Gateway providers, credentials, and models |
| `/v3/evaluators` | GET | List all evaluators in the org |
| `/v3/evaluators` | POST | Create a new evaluator |
| `/v3/evaluators/{id}` | GET | Get evaluator details |
| `/v3/evaluators/{id}` | PATCH | Update an evaluator |
| `/v3/evaluators/{id}/archive` | POST | Archive an evaluator |
| `/v3/evaluator-rules` | GET | List evaluator rules (supports filtering) |
| `/v3/evaluator-rules` | POST | Create an evaluator rule (bind evaluator to app) |
| `/v3/evaluator-rules/{id}` | PATCH | Update an evaluator rule |
| `/v3/evaluator-rules/{id}` | DELETE | Delete an evaluator rule |
| `/v3/evaluator-rules/map-input-keys` | POST | Discover input fields and available span attributes |
| `/v3/genai-enrichment-backfills` | GET | List backfill job statuses |

## Further Reading

- [Evaluator Rules](https://docs.fiddler.ai/evaluate-and-test/evaluator-rules) -- Setup, input mapping, and backfill for production evaluator rules
- [LLM Evaluation Prompt Specs](https://docs.fiddler.ai/observability/llm/llm-evaluation-prompt-specs) -- Deep dive on `prompt_spec`, `prompt_template`, and `output_fields` for custom judges
- [LLM Observability Metrics Reference](https://docs.fiddler.ai/reference/llm-observability-metrics) -- Complete catalog of all evaluator/enrichment types, output columns, and LLM requirements